<a href="https://colab.research.google.com/github/Lucas66666677/Simple-Automatic-English-TA/blob/main/%E2%80%9C%E8%8B%B1%E6%96%87%E5%82%99%E8%AA%B2%E5%8A%A9%E6%89%8B%EF%BC%88Groq_API%EF%BC%89%E2%80%9D.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##安裝依賴套件

In [1]:
# --- 1. 安裝環境與依賴套件 ---
import os

# 安裝系統級工具 Poppler (PDF 轉圖片的核心)
os.system("apt-get update -y && apt-get install -y poppler-utils")

# 安裝 Python 庫
os.system("pip install --upgrade langchain-groq langchain langchain-community langchain-core gradio pypdf markdown pdf2image")

# --- 2. 引入套件 ---
import sys
import time
import base64
import gradio as gr
import pypdf
import io
from pdf2image import convert_from_path
from langchain_groq import ChatGroq
from google.colab import userdata
from langchain_core.prompts import PromptTemplate
from langchain_core.messages import HumanMessage
from langchain_core.output_parsers import StrOutputParser

# 創建臨時資料夾避免路徑錯誤
TEMP_DIR = "/tmp/lecturas_temp"
os.makedirs(TEMP_DIR, exist_ok=True)
print("環境與套件準備完成！")

環境與套件準備完成！


##執行 Lecturas 系統

##讀取 API Key

In [2]:
try:
    GROQ_API_KEY = userdata.get('Groq')
except Exception:
    GROQ_API_KEY = None

##輔助函式：圖片轉 Base64

In [3]:
def encode_image_base64(pil_image):
    """將 PIL 圖片轉為 Base64 字串供 AI 讀取"""
    buffered = io.BytesIO()
    pil_image.save(buffered, format="JPEG")
    return base64.b64encode(buffered.getvalue()).decode('utf-8')

##定義核心處理函式

In [ ]:
def process_lecturas(file_obj, text_input, image_obj):

    if not GROQ_API_KEY:
        return "❌ 錯誤：找不到 API Key", "❌ 請設定 Secrets", "Key Missing"

    try:
        # 文字大腦 (Llama 3.3)
        llm_text = ChatGroq(temperature=0.3, model_name="llama-3.3-70b-versatile", groq_api_key=GROQ_API_KEY)
        # 視覺大腦 (Llama 4 Vision)
        llm_vision = ChatGroq(temperature=0.1, model_name="meta-llama/llama-4-scout-17b-16e-instruct", groq_api_key=GROQ_API_KEY)
    except Exception as e:
        return f"連線錯誤: {e}", "連線錯誤", str(e)

    vision_output = ""
    clean_text = ""
    debug_log = ""

    # --- A. 圖片題目處理 (通用版) ---
    if image_obj:
        try:
            import PIL.Image
            if isinstance(image_obj, str):
                img = PIL.Image.open(image_obj)
            else:
                img = image_obj
            b64_img = encode_image_base64(img)

            # 視覺 Prompt 改為通用型
            msg = HumanMessage(content=[
                {"type": "text", "text": "你是一位全科資深教師。請分析這張圖片（可能是題目、圖表或文章），提供「老師版詳解」。如果是數理題，請寫出計算步驟；如果是文科題，請解析文章重點或圖片含義。"},
                {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{b64_img}"}}
            ])
            res = llm_vision.invoke([msg])
            vision_output = f"### 📸 圖片解析\n{res.content}\n\n---\n\n"
        except Exception as e:
            vision_output = f"⚠️ 圖片解析失敗: {e}\n\n"

    # --- B. PDF 處理 ---
    source_type = "無"
    if file_obj:
        try:
            reader = pypdf.PdfReader(file_obj.name)
            for page in reader.pages:
                t = page.extract_text()
                if t: clean_text += t + "\n"
            source_type = "PDF (文字層)"
        except:
            pass

        if len(clean_text) < 50:
            debug_log += "⚠️ 啟動視覺辨識 (OCR)...\n"
            source_type = "PDF (視覺辨識)"
            try:
                images = convert_from_path(file_obj.name, first_page=1, last_page=1, output_folder=TEMP_DIR)
                if images:
                    b64_pdf_img = encode_image_base64(images[0])
                    msg = HumanMessage(content=[
                        {"type": "text", "text": "請將圖片中所有的文字內容完整轉錄出來。"},
                        {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{b64_pdf_img}"}}
                    ])
                    ocr_res = llm_vision.invoke([msg])
                    clean_text = ocr_res.content
                    debug_log += "✅ 視覺讀取成功！\n"
            except Exception as e:
                debug_log += f"❌ 視覺讀取失敗: {e}\n"

    if not clean_text and text_input:
        clean_text = text_input

    debug_log += f"最終來源: {source_type} | 字數: {len(clean_text)}"

    # --- D. 生成結果 (核心修改：智慧學科判斷 Prompt) ---
    final_left = ""
    final_right = vision_output

    if len(clean_text) >= 10:
        # Prompt A: 板書 (智慧判斷)
        prompt_a = PromptTemplate(
            template="""
            <Role>你是一位博學多聞的資深教師，精通各個學科。</Role>
            <Task>請先判斷【教材文本】的學科類型，然後根據該學科的特性，整理出一份「結構化板書」。</Task>

            <Subject_Rules>
            1. **如果是【數理科】(物理/數學/化學)**：
               - 必須寫出 **核心公式** (使用易讀符號如 F=ma, x^2，禁止 LaTeX)。
               - 必須標註 **變數定義** 與 **計算陷阱**。

            2. **如果是【人文科】(歷史/地理/公民)**：
               - 歷史：整理 **時間軸 (Timeline)**、**因果關係 (Cause & Effect)**、**關鍵人物**。
               - 地理：整理 **位置特徵**、**氣候/地形成因**、**經濟活動**。
               - 公民：整理 **法律原則**、**專有名詞定義**、**正反方觀點**。

            3. **如果是【語文科】(國文/英文)**：
               - 國文：整理 **修辭手法**、**形音義辨正**、**課文主旨**、**名句賞析**。
               - 英文：整理 **關鍵單字 (Vocabulary)**、**文法句型 (Grammar)**、**易混淆詞**。
            </Subject_Rules>

            <Format>
            Markdown 巢狀清單：
            * **[核心主題/章節]**
              * **重點一**：(根據學科特性的內容)
              * 👉 **考點/陷阱**：(該學科的易錯點)
            </Format>

            <Content>
            {text}
            </Content>
            """,
            input_variables=["text"]
        )

        # Prompt B: 腳本 (智慧判斷)
        prompt_b = PromptTemplate(
            template="""
            <Role>你是一位擅長引導的補習班名師。</Role>
            <Task>根據教材學科，設計一份「口語化」的教學腳本。</Task>

            <Scenario_Adaptation>
            * **數理科** -> 著重「放聲思考」解題步驟 (第一步想什麼、第二步做什麼)。
            * **人文科** -> 著重「說故事」與「脈絡連結」(為什麼會發生這件事？後來怎麼了？)。
            * **語文科** -> 著重「語感培養」與「例句示範」。
            </Scenario_Adaptation>

            <Structure>
            ### 🎭 Part 1: 引發動機 (Hook)
            * 用生活例子或有趣提問開場。

            ### 🧠 Part 2: 核心觀念引導 (Deep Dive)
            * 蘇格拉底式提問，引導學生思考核心概念。

            ### 💡 Part 3: 實戰演練 (Practice)
            * 數理：帶一題計算。
            * 人文/語文：帶一個案例分析或句子練習。
            </Structure>

            <Content>
            {text}
            </Content>
            """,
            input_variables=["text"]
        )

        try:
            chain_a = prompt_a | llm_text | StrOutputParser()
            chain_b = prompt_b | llm_text | StrOutputParser()

            res_a = chain_a.invoke({"text": clean_text})
            res_b = chain_b.invoke({"text": clean_text})

            final_left = res_a
            final_right += res_b
        except Exception as e:
            final_left = f"生成錯誤: {e}"
    else:
        final_left = "⚠️ 無法讀取內容"

    return final_left, final_right, debug_log

# ==========================================
# 5. 啟動 UI
# ==========================================
with gr.Blocks(title="Lecturas v4.0 (Universal)", theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 📖 Lecturas —— 備課助手 (v4.0 全科通用版)")
    gr.Markdown("✅ 自動識別學科：數理 (公式)、歷史 (時間軸)、英文 (文法/單字)。")

    with gr.Row():
        with gr.Column(scale=1):
            file_input = gr.File(label="📄 1. 上傳教材 (任何科目 PDF)", file_types=[".pdf"])
            text_input = gr.Textbox(label="或是貼上文字", lines=2)
        with gr.Column(scale=1):
            image_input = gr.Image(type="filepath", label="📸 2. 上傳圖片 (題目/文章/圖表)", height=200)

    btn = gr.Button("🚀 智慧分析 (Generate)", variant="primary")

    gr.Markdown("---")

    with gr.Row():
        out1 = gr.Markdown(label="板書 (依學科自動調整)")
        out2 = gr.Markdown(label="腳本 (口語化)")

    with gr.Accordion("Debug Log", open=False):
        debug_out = gr.Textbox()

    btn.click(process_lecturas, inputs=[file_input, text_input, image_input], outputs=[out1, out2, debug_out])

print("正在啟動...")
demo.launch(debug=True)

/tmp/ipykernel_4927/730667032.py:162: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(title="Lecturas v4.0 (Universal)", theme=gr.themes.Soft()) as demo:


正在啟動...
It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://eeb4c0eabce5ff5001.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
